# General Code

In [221]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

In [222]:
df = pd.read_csv('raw/AOIBaseTFG.tsv', sep='\t')


claro = df[df['Timeline'] == 'Timeline1-Mode1']
oscuro = df[df['Timeline'] == 'Timeline2-Mode2']


## Basic functions

In [ ]:
def obtainBasicData(array):
    mean = np.mean(array)
    
    std = np.std(array)
    max = np.max(array)
    min = np.min(array)
    return mean, std, max, min

def printBasicData(array, label):
    mean, std, max, min = obtainBasicData(array)
    print(f"\n{label} - Tiempo hasta la primera fijación")
    print(f"Mean: {mean:.2f},  Std: {std:.2f}, Max: {max:.2f}, Min: {min:.2f}")

# Test function i made to understand how to do 
def ttFirstFixation(aoi, toiClaro = 'Entire Recording', toiOscuro = 'Entire Recording'):
    ttf_claro = claro[claro['AOI'] == aoi]
    ttf_claro = ttf_claro[ttf_claro['TOI'] == toiClaro]['Time_to_first_fixation']
    
    ttf_oscuro = oscuro[oscuro['AOI'] == aoi]
    ttf_oscuro = ttf_oscuro[ttf_oscuro['TOI'] == toiOscuro]['Time_to_first_fixation']

    printBasicData(ttf_claro, f"{aoi} Claro")
    printBasicData(ttf_oscuro, f"{aoi} Oscuro")
    print("\n-------------------------")
    print("Apply test t-student for independent samples:")
    tStat, p_value = stats.ttest_ind(ttf_claro, ttf_oscuro, equal_var=False)
    print(f"T-statistic: {tStat:.4f}, P-value: {p_value:.4f}")
    if(p_value < 0.05):
        print("The difference is statistically significant.")
    return ttf_claro, ttf_oscuro

def timeToFirstFixationPlot(ttf_claro, ttf_oscuro, aoi):
    labels = ['Claro', 'Oscuro']
    data = [ttf_claro, ttf_oscuro]

    plt.figure(figsize=(8, 6))
    plt.boxplot(data, labels=labels)
    plt.title(f'Time to First Fixation for AOI: {aoi}')
    plt.ylabel('Time to First Fixation (ms)')
    plt.grid(axis='y')
    plt.show()
    
def saveToCSV(dataDict, filename):
    with open("output/" + filename, 'w') as f:
        f.write("AOI,Claro Mean,Claro Std,Claro Max,Claro Min,Claro Count,Oscuro Mean,Oscuro Std,Oscuro Max,Oscuro Min,Oscuro Count,T-stat,P-value,Significant\n")
        for aoi, data in dataDict.items():
            f.write(f"{aoi},{data.claroMean:.3f},{data.claroStd:.3f},{data.claroMax:.3f},{data.claroMin:.3f},{data.claroCount},{data.oscuroMean:.3f},{data.oscuroStd:.3f},{data.oscuroMax:.3f},{data.oscuroMin:.3f},{data.oscuroCount},{data.tStat:.5f},{data.pValue:.5f},Yes\n")


## Bulk function

In [ ]:
class TimeToFirstFixationObject:
    claroMean: float
    claroStd: float
    oscuroMean: float
    oscuroStd: float
    aoi: str
    pValue: float
    tStat: float
    claroMax: float
    claroMin: float
    oscuroMax: float
    oscuroMin: float
    oscuroCount: int
    claroCount: int
    

def timeToFirstFixationBulk(column, toiClaro = 'Entire Recording', toiOscuro = 'Entire Recording',csv= True, csvName = "bulk.csv"):
    claro_bulk = df[df['Timeline'] == 'Timeline1-Mode1']
    oscuro_bulk = df[df['Timeline'] == 'Timeline2-Mode2']
    
    if column.dtype == str:
        claro_bulk[column] = pd.to_numeric(claro_bulk[column].replace(',','.'), errors='coerce')
        oscuro_bulk[column] = pd.to_numeric(oscuro_bulk[column].replace(',', '.'), errors='coerce')
    
    
    claro_bulk = claro_bulk[claro_bulk[column] > 0] # It can make noise in the data
    oscuro_bulk = oscuro_bulk[oscuro_bulk[column] > 0] 

    aois = df['AOI'].unique()
    significativeResults = {}
    noSignificativeResults = {}
    
    for aoi in aois:
        aoiObject = TimeToFirstFixationObject()
        ttf_claro = claro_bulk[claro_bulk['AOI'] == aoi]
        ttf_claro = ttf_claro[ttf_claro['TOI'] == toiClaro][column]
        ttf_claro = ttf_claro.dropna()
        
        ttf_oscuro = oscuro_bulk[oscuro_bulk['AOI'] == aoi]
        ttf_oscuro = ttf_oscuro[ttf_oscuro['TOI'] == toiOscuro][column]
        ttf_oscuro = ttf_oscuro.dropna()
        
        # Skip if either dataset is empty
        if len(ttf_claro) == 0 or len(ttf_oscuro) == 0:
            continue
            
        aoiObject.claroMean, aoiObject.claroStd, aoiObject.claroMax, aoiObject.claroMin = obtainBasicData(ttf_claro)
        aoiObject.oscuroMean, aoiObject.oscuroStd, aoiObject.oscuroMax, aoiObject.oscuroMin = obtainBasicData(ttf_oscuro)
        aoiObject.aoi = aoi
        
        tStat, p_value = stats.ttest_ind(ttf_claro, ttf_oscuro, equal_var=False)
        aoiObject.pValue = p_value
        aoiObject.tStat = tStat
        aoiObject.claroCount = len(ttf_claro)
        aoiObject.oscuroCount = len(ttf_oscuro)
        if(p_value < 0.05):
            significativeResults[aoi] = aoiObject
        else:
            noSignificativeResults[aoi] = aoiObject
        
    if csv:
        saveToCSV(significativeResults, "sig_" + csvName)
        saveToCSV(noSignificativeResults, "no_sig_" + csvName)

    return significativeResults, noSignificativeResults

# Time to first Fixation

In [226]:
claro = claro[claro['Time_to_first_fixation'] > 0] # It can make noise in the data
oscuro = oscuro[oscuro['Time_to_first_fixation'] > 0] # It can make noise in the data

### Patata

In [227]:
ttf_titulo_claro, ttf_titulo_oscuro = ttFirstFixation('Titulo')


Titulo Claro - Tiempo hasta la primera fijación
Mean: 112888.90,  Std: 20551.21, Max: 149679.00, Min: 85900.00

Titulo Oscuro - Tiempo hasta la primera fijación
Mean: 114903.50,  Std: 23804.81, Max: 180827.00, Min: 87543.00

-------------------------
Apply test t-student for independent samples:
T-statistic: -0.2117, P-value: 0.8343


# TimeToFirstFixation Bulk

In [ ]:
import os

significativeResults, noSignificativeResults = timeToFirstFixationBulk('Patata')





with open('output/significative_patata.csv', 'w') as f:
    f.write("AOI,Claro Mean,Claro Std,Claro Max,Claro Min,Claro Count,Oscuro Mean,Oscuro Std,Oscuro Max,Oscuro Min,Oscuro Count,T-stat,P-value,Significant\n")
    for aoi, data in significativeResults.items():
        f.write(f"{aoi},{data.claroMean:.2f},{data.claroStd:.2f},{data.claroMax:.2f},{data.claroMin:.2f},{data.claroCount},{data.oscuroMean:.2f},{data.oscuroStd:.2f},{data.oscuroMax:.2f},{data.oscuroMin:.2f},{data.oscuroCount},{data.tStat:.4f},{data.pValue:.4f},Yes\n")


with open('output/noSignificative_patata.csv', 'w') as f:
    f.write("AOI,Claro Mean,Claro Std,Claro Max,Claro Min,Claro Count,Oscuro Mean,Oscuro Std,Oscuro Max,Oscuro Min,Oscuro Count,T-stat,P-value,Significant\n")
    for aoi, data in noSignificativeResults.items():
        f.write(f"{aoi},{data.claroMean:.2f},{data.claroStd:.2f},{data.claroMax:.2f},{data.claroMin:.2f},{data.claroCount},{data.oscuroMean:.2f},{data.oscuroStd:.2f},{data.oscuroMax:.2f},{data.oscuroMin:.2f},{data.oscuroCount},{data.tStat:.4f},{data.pValue:.4f},No\n")

print(f"Saved {len(significativeResults)} significative results to output/significative_patata.csv")
print(f"Saved {len(noSignificativeResults)} non-significative results to output/noSignificative_patata.csv")


Saved 8 significative results to output/significative_patata.csv
Saved 197 non-significative results to output/noSignificative_patata.csv


In [ ]:
significativeResults, noSignificativeResults = timeToFirstFixationBulk('Mochila')

with open('output/significative_mochila.csv', 'w') as f:
    f.write("AOI,Claro Mean,Claro Std,Claro Max,Claro Min,Claro Count,Oscuro Mean,Oscuro Std,Oscuro Max,Oscuro Min,Oscuro Count,T-stat,P-value,Significant\n")
    for aoi, data in significativeResults.items():
        f.write(f"{aoi},{data.claroMean:.2f},{data.claroStd:.2f},{data.claroMax:.2f},{data.claroMin:.2f},{data.claroCount},{data.oscuroMean:.2f},{data.oscuroStd:.2f},{data.oscuroMax:.2f},{data.oscuroMin:.2f},{data.oscuroCount},{data.tStat:.4f},{data.pValue:.4f},Yes\n")


with open('output/noSignificative_mochila.csv', 'w') as f:
    f.write("AOI,Claro Mean,Claro Std,Claro Max,Claro Min,Claro Count,Oscuro Mean,Oscuro Std,Oscuro Max,Oscuro Min,Oscuro Count,T-stat,P-value,Significant\n")
    for aoi, data in noSignificativeResults.items():
        f.write(f"{aoi},{data.claroMean:.2f},{data.claroStd:.2f},{data.claroMax:.2f},{data.claroMin:.2f},{data.claroCount},{data.oscuroMean:.2f},{data.oscuroStd:.2f},{data.oscuroMax:.2f},{data.oscuroMin:.2f},{data.oscuroCount},{data.tStat:.4f},{data.pValue:.4f},No\n")

print(f"Saved {len(significativeResults)} significative results to output/significative_mochila.csv")
print(f"Saved {len(noSignificativeResults)} non-significative results to output/noSignificative_mochila.csv")

Saved 25 significative results to output/significative_mochila.csv
Saved 312 non-significative results to output/noSignificative_mochila.csv


In [ ]:
claro_text = df[df['Timeline'] == 'Timeline1-Mode1']
oscuro_text = df[df['Timeline'] == 'Timeline2-Mode2']

claro_bulk = claro_text[claro_text['Time_to_first_fixation'] > 0] # It can make noise in the data
oscuro_bulk = oscuro_text[oscuro_text['Time_to_first_fixation'] > 0]

aois = df['AOI'].unique()
significativeResults = {}
noSignificativeResults = {}
    
for aoi in aois:
    aoiObject = TimeToFirstFixationObject()
    ttf_claro = claro_bulk[claro_bulk['AOI'] == aoi]
    ttf_claro = ttf_claro[ttf_claro['TOI'] == "Text1"]['Time_to_first_fixation']
    ttf_claro = ttf_claro.dropna()
        
    ttf_oscuro = oscuro_bulk[oscuro_bulk['AOI'] == aoi]
    ttf_oscuro = ttf_oscuro[ttf_oscuro['TOI'] == "Text"]['Time_to_first_fixation']
    ttf_oscuro = ttf_oscuro.dropna()
        
    # Skip if either dataset is empty
    if len(ttf_claro) == 0 or len(ttf_oscuro) == 0:
        continue
            
    aoiObject.claroMean, aoiObject.claroStd, aoiObject.claroMax, aoiObject.claroMin = obtainBasicData(ttf_claro)
    aoiObject.oscuroMean, aoiObject.oscuroStd, aoiObject.oscuroMax, aoiObject.oscuroMin = obtainBasicData(ttf_oscuro)
    aoiObject.aoi = aoi
        
    tStat, p_value = stats.ttest_ind(ttf_claro, ttf_oscuro, equal_var=False)
    aoiObject.pValue = p_value
    aoiObject.tStat = tStat
    aoiObject.claroCount = len(ttf_claro)
    aoiObject.oscuroCount = len(ttf_oscuro)
    if(p_value < 0.05):
        significativeResults[aoi] = aoiObject
    else:
        noSignificativeResults[aoi] = aoiObject
        
with open('output/significative_text.csv', 'w') as f:
    f.write("AOI,Claro Mean,Claro Std,Claro Max,Claro Min,Claro Count,Oscuro Mean,Oscuro Std,Oscuro Max,Oscuro Min,Oscuro Count,T-stat,P-value,Significant\n")
    for aoi, data in significativeResults.items():
        f.write(f"{aoi},{data.claroMean:.2f},{data.claroStd:.2f},{data.claroMax:.2f},{data.claroMin:.2f},{data.claroCount},{data.oscuroMean:.2f},{data.oscuroStd:.2f},{data.oscuroMax:.2f},{data.oscuroMin:.2f},{data.oscuroCount},{data.tStat:.4f},{data.pValue:.4f},Yes\n")


with open('output/noSignificative_text.csv', 'w') as f:
    f.write("AOI,Claro Mean,Claro Std,Claro Max,Claro Min,Claro Count,Oscuro Mean,Oscuro Std,Oscuro Max,Oscuro Min,Oscuro Count,T-stat,P-value,Significant\n")
    for aoi, data in noSignificativeResults.items():
        f.write(f"{aoi},{data.claroMean:.2f},{data.claroStd:.2f},{data.claroMax:.2f},{data.claroMin:.2f},{data.claroCount},{data.oscuroMean:.2f},{data.oscuroStd:.2f},{data.oscuroMax:.2f},{data.oscuroMin:.2f},{data.oscuroCount},{data.tStat:.4f},{data.pValue:.4f},No\n")

print(f"Saved {len(significativeResults)} significative results to output/significative_text.csv")
print(f"Saved {len(noSignificativeResults)} non-significative results to output/noSignificative_text.csv")

Saved 0 significative results to output/significative_text.csv
Saved 41 non-significative results to output/noSignificative_text.csv
